In [1]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
df.shape

(50000, 2)

In [3]:
df.drop_duplicates(inplace=True)

In [4]:
df.head()
df.shape

(49582, 2)

# Pre - Processing

## 1. Converting to lowercase

In [5]:
df["review"] = df["review"].str.lower()

## 2. Removing the urls

In [6]:
import re

def remove_urls(text):
    text = re.sub(r"https\S+", "", text)
    return text
df["review"] = df["review"].apply(remove_urls)

## 3. Removing punctuations

In [7]:
def remove_punctuation(text):
    text = re.sub(r"[^A-za-z0-9\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punctuation)

## 4. Removing HTML

In [8]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

## 5. Removing Stopwords

In [9]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# def remove_stopwords(text):
#     tokens = word_tokenize(text)
#     stop_words = stopwords.words("english")

#     for word in tokens:
#         if word in stop_words:
#             text = text.replace(word, "")
#     return text

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    token = word_tokenize(text)
    filtered = [word for word in token if word not in stop_words]

    return " ".join(filtered)


df["review"] = df["review"].apply(remove_stopwords)

In [11]:
df["review"]

0        one reviewers mentioned watching 1 oz episode ...
1        wonderful little production br br filming tech...
2        thought wonderful way spend time hot summer we...
3        basically theres family little boy jake thinks...
4        petter matteis love time money visually stunni...
                               ...                        
49995    thought movie right good job wasnt creative or...
49996    bad plot bad dialogue bad acting idiotic direc...
49997    catholic taught parochial elementary schools n...
49998    im going disagree previous comment side maltin...
49999    one expects star trek movies high art fans exp...
Name: review, Length: 49582, dtype: str

## 6. Stemming

In [12]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()

def stemming(text):
    return " ".join(ps.stem(token) for token in word_tokenize(text))

df["review"] = df["review"].apply(stemming)

In [13]:
df["review"]

0        one review mention watch 1 oz episod youll hoo...
1        wonder littl product br br film techniqu unass...
2        thought wonder way spend time hot summer weeke...
3        basic there famili littl boy jake think there ...
4        petter mattei love time money visual stun film...
                               ...                        
49995    thought movi right good job wasnt creativ orig...
49996    bad plot bad dialogu bad act idiot direct anno...
49997    cathol taught parochi elementari school nun ta...
49998    im go disagre previou comment side maltin one ...
49999    one expect star trek movi high art fan expect ...
Name: review, Length: 49582, dtype: str

## 7. Encoding

In [14]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df["sentiment"])

y

array([1, 1, 1, ..., 0, 0, 0], shape=(49582,))

## 8. Vectorization

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4107854 stored elements and shape (49582, 5000)>

# Dataset and DataLoader

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
import torch
from torch.utils.data import DataLoader, TensorDataset

X_train = X_train.toarray()
X_test = X_test.toarray()

In [18]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test).float()
)

In [19]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

# BUILD RNN

In [20]:
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size = 128, num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):

        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)

        out = self.fc(out[:, -1, :])
        return out

input_size = X_train.shape[1]
model = RNN(input_size).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [21]:
epochs = 100

for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        xb = xb.unsqueeze(1)

        out = model(xb)
        out = torch.sigmoid(out.squeeze())

        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

    print(f"Epoch = {epoch+1} \ {epochs}  and Loss = {loss.item()}")

Epoch = 1 \ 100  and Loss = 0.24217496812343597
Epoch = 2 \ 100  and Loss = 0.291177362203598
Epoch = 3 \ 100  and Loss = 0.45286980271339417
Epoch = 4 \ 100  and Loss = 0.2532386779785156
Epoch = 5 \ 100  and Loss = 0.14656101167201996
Epoch = 6 \ 100  and Loss = 0.6048769354820251
Epoch = 7 \ 100  and Loss = 0.13809934258460999
Epoch = 8 \ 100  and Loss = 0.24588365852832794
Epoch = 9 \ 100  and Loss = 0.17001917958259583
Epoch = 10 \ 100  and Loss = 0.33352944254875183
Epoch = 11 \ 100  and Loss = 0.11470065265893936
Epoch = 12 \ 100  and Loss = 0.14028765261173248
Epoch = 13 \ 100  and Loss = 0.20890311896800995
Epoch = 14 \ 100  and Loss = 0.20011012256145477
Epoch = 15 \ 100  and Loss = 0.09136924892663956
Epoch = 16 \ 100  and Loss = 0.1254492998123169
Epoch = 17 \ 100  and Loss = 0.2104450762271881
Epoch = 18 \ 100  and Loss = 0.2608761489391327
Epoch = 19 \ 100  and Loss = 0.2573590576648712
Epoch = 20 \ 100  and Loss = 0.2990376651287079
Epoch = 21 \ 100  and Loss = 0.2745437

In [22]:
model.eval()

with torch.no_grad():
    correct_val = 0
    tot_val = 0
    for xb, yb in test_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        xb = xb.unsqueeze(1)
        out = model(xb)
        pred = (torch.sigmoid(out.squeeze()) > 0.5).float()
        tot_val += yb.size(0)
        correct_val += (pred == yb).sum().item()

    print(f"Accuracy = {correct_val/tot_val*100}")

Accuracy = 86.1752546132903
